# vton_pipeline.py — 반려동물 가상 피팅(VTON) AI 파이프라인

## 이 파일은 무엇인가요?

이 파일은 프로젝트의 **"두뇌(Brain)"** 역할을 하는 핵심 AI 파이프라인입니다.

`main.py`(API 서버)가 외부 요청을 받아서 이 파이프라인에게 작업을 넘기면, 이 파일 안의 코드가 **3개의 AI 모델을 순서대로 실행**하여 반려동물 사진에 옷을 입힌 합성 이미지를 만들어냅니다.

### 사용하는 AI 모델 3가지

| 순서 | AI 모델 | 제작사 | 역할 | 오픈소스 여부 |
|---|---|---|---|---|
| **1단계** | **AutoML (Vertex AI)** | Google Cloud | 반려동물이 사진 속 어디에 있는지 **위치(네모 박스)**를 찾습니다. | 유료 (GCP 서비스) |
| **2단계** | **SAM (Segment Anything Model)** | Meta (Facebook) | 네모 박스 안에서 반려동물의 정확한 **실루엣(윤곽선/누끼)을 픽셀 단위로** 따냅니다. | 무료 오픈소스 |
| **3단계** | **Stable Diffusion + IP-Adapter** | RunwayML + Tencent | 따낸 실루엣 영역에 **옷의 색상, 패턴, 질감을 자연스럽게 합성**합니다. | 무료 오픈소스 |

### 전체 처리 흐름 (시각화)

```
[반려동물 사진] + [옷 사진]
        |                |
        v                |
  1. AutoML → 바운딩 박스 (네모 박스 좌표)
        |                |
        v                |
  2. SAM → 마스크 (반려동물 실루엣, 흰/검 이미지)
        |                |
        v                v
  3. Stable Diffusion (Inpainting + IP-Adapter)
        |
        v
  [옷을 입은 반려동물 합성 이미지] → JPEG 바이트로 변환 → main.py로 반환
```

## 1단계: 라이브러리 불러오기 (Import)

- `cv2` (OpenCV): 이미지를 읽고, 색 공간을 변환(BGR↔RGB)하고, 바이트 데이터를 이미지로 디코딩하는 컴퓨터 비전 라이브러리입니다.
- `numpy (np)`: 이미지를 숫자 배열(행렬)로 다루기 위한 수학 라이브러리입니다. 모든 AI 모델은 이미지를 numpy 배열로 변환해서 입력받습니다.
- `torch` (PyTorch): Meta가 만든 딥러닝 프레임워크입니다. SAM과 Stable Diffusion 모두 PyTorch 위에서 동작합니다. GPU(CUDA)가 있으면 자동으로 GPU를 사용하여 연산을 가속합니다.
- `io.BytesIO`: 파이썬에서 바이트 데이터를 파일처럼 다룰 수 있게 해주는 도구입니다. 결과 이미지를 JPEG 바이트로 변환할 때 사용합니다.
- `aiplatform` (Vertex AI): Google Cloud의 AI 플랫폼 서비스에 접속하여 AutoML 모델을 호출하는 도구입니다.
- `segment_anything`: Meta가 공개한 SAM(Segment Anything Model)의 파이썬 패키지입니다.
  - `sam_model_registry`: SAM 모델의 크기별 버전(vit_b, vit_l, vit_h)을 선택하는 딕셔너리입니다.
  - `SamPredictor`: 이미지와 좌표를 넣으면 마스크(실루엣)를 예측해주는 추론 도구입니다.
- `diffusers`: Hugging Face에서 만든 Stable Diffusion 전용 파이썬 패키지입니다.
  - `StableDiffusionInpaintPipeline`: 이미지의 특정 영역(마스크)을 지우고 그 자리에 새로운 이미지를 채워넣는 Inpainting 전용 파이프라인입니다.
- `PIL.Image`: 이미지를 열고, 저장하고, 포맷을 변환하는 가장 기본적인 파이썬 이미지 처리 라이브러리입니다.

In [ ]:
import cv2
import numpy as np
import torch
import io
from google.cloud import aiplatform
from segment_anything import sam_model_registry, SamPredictor
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image

## 2단계: PetVTONPipeline 클래스 — `__init__` (초기화/모델 로딩)

서버가 시작될 때 **딱 한 번만** 실행되는 초기화 코드입니다.
3개의 무거운 AI 모델을 메모리(RAM 또는 GPU VRAM)에 올려놓습니다.

> **왜 한 번만 로드하나요?** AI 모델 파일의 크기는 수 GB에 달합니다. 요청이 올 때마다 매번 로드하면 몇 분씩 걸리므로, 서버 시작 시 한 번만 로드하고 메모리에 유지합니다.

### 2-1. 디바이스(Device) 설정
```python
self.device = "cuda" if torch.cuda.is_available() else "cpu"
```
- `cuda`: NVIDIA GPU가 있으면 GPU를 사용합니다. (수십 배 빠름)
- `cpu`: GPU가 없으면 CPU를 사용합니다. (느리지만 동작은 가능)

### 2-2. SAM 모델 로드
- `sam_model_registry["vit_h"]`: SAM의 가장 큰 버전(ViT-H, 약 2.4GB)을 선택합니다.
  - `vit_b` (가장 작음/빠름) < `vit_l` (중간) < `vit_h` (가장 큼/정확)
- `checkpoint`: 로컬 `weights/` 폴더에 미리 다운로드해 둔 가중치 파일의 경로입니다.
- `SamPredictor(sam)`: SAM 모델을 감싸서 "이미지 + 좌표 → 마스크"를 쉽게 추론할 수 있게 해주는 래퍼(Wrapper)입니다.

### 2-3. Stable Diffusion Inpainting 모델 로드
- `from_pretrained("./weights/sd-inpainting-model")`: 로컬 폴더에서 사전학습된 Stable Diffusion 모델을 불러옵니다.
- `local_files_only=True`: 인터넷에서 모델을 다운로드하지 않고 로컬 파일만 사용합니다. (오프라인 환경 대응)
- `torch_dtype`: GPU가 있으면 `float16`(메모리 절약, 속도 향상), CPU면 `float32`(안정적)로 설정합니다.

### 2-4. IP-Adapter 장착
IP-Adapter는 Stable Diffusion의 **플러그인**입니다. 기본 Stable Diffusion은 텍스트(프롬프트)만 입력받을 수 있지만, IP-Adapter를 장착하면 **이미지도 프롬프트로 입력**할 수 있게 됩니다.

- `load_ip_adapter()`: Stable Diffusion 파이프라인에 IP-Adapter 가중치를 끼워넣습니다.
- `set_ip_adapter_scale(0.8)`: 옷 이미지가 결과물에 미치는 영향력(강도)을 설정합니다.
  - `0.0`: 옷 이미지를 완전히 무시 (텍스트 프롬프트만 반영)
  - `1.0`: 옷 이미지를 최대한 반영 (텍스트 프롬프트를 거의 무시)
  - `0.8`: 옷 이미지의 영향이 매우 강하게 반영됨
- `safety_checker = None`: Stable Diffusion의 안전 필터(NSFW 검출기)를 비활성화합니다. 반려동물 사진은 민감한 콘텐츠가 아니므로 불필요합니다.

### 2-5. GCP AutoML (Vertex AI) 초기화
- `aiplatform.init()`: GCP 프로젝트와 리전(서버 위치)을 설정합니다.
- `Endpoint("88190178396471296")`: Google Cloud에 배포해 둔 AutoML 객체 탐지 모델의 엔드포인트 ID입니다. 이 엔드포인트에 이미지를 보내면, 반려동물의 위치(바운딩 박스 좌표)를 응답으로 받습니다.

In [ ]:
class PetVTONPipeline:
    def __init__(self):
        print("AI 모델 로딩 시작...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        # 1. SAM 모델 로드 (로컬 폴더에서)
        sam_checkpoint = "./weights/sam_vit_h_4b8939.pth"
        sam = sam_model_registry["vit_h"](checkpoint=sam_checkpoint)
        sam.to(device=self.device)
        self.sam_predictor = SamPredictor(sam)

        # 2. Diffusion 모델 로드 (로컬 폴더에서 오프라인 모드로)
        self.diffusion_pipe = StableDiffusionInpaintPipeline.from_pretrained(
            "./weights/sd-inpainting-model",
            local_files_only=True,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32
        ).to(self.device)

        # IP-Adapter 플러그인 장착
        self.diffusion_pipe.load_ip_adapter(
            "./weights/IP-Adapter",
            subfolder="models", 
            weight_name="ip-adapter_sd15.bin"
        )

        # 옷 사진(이미지 프롬프트)이 결과물에 미치는 영향력(강도) 조절 (0.0 ~ 1.0)
        self.diffusion_pipe.set_ip_adapter_scale(0.8) 
        
        # 안전 필터 해제
        self.diffusion_pipe.safety_checker = None

        # 3. GCP AutoML (Vertex AI) 초기화
        aiplatform.init(project="knu-2026-bangjeongho833", location="us-central1")
        self.automl_endpoint = aiplatform.Endpoint("88190178396471296")
        
        print("모든 AI 모델 로딩 완료!")

## 3단계: `_get_bounding_box` — AutoML로 반려동물 위치 탐지

이 메서드는 반려동물 이미지를 **GCP AutoML에 보내서**, 반려동물이 사진 속 **어디에 있는지** 네모 박스(바운딩 박스) 좌표를 받아오는 역할입니다.

### 바운딩 박스(Bounding Box)란?
사진 속 물체(여기서는 반려동물)를 감싸는 **최소 크기의 직사각형**입니다. 4개의 숫자로 표현됩니다:

```
(x_min, y_min) ──────────────┐
     │                       │
     │    [반려동물 영역]     │
     │                       │
     └──────────────── (x_max, y_max)
```

- `x_min, y_min`: 네모 박스의 왼쪽 위 꼭짓점 좌표 (픽셀 단위)
- `x_max, y_max`: 네모 박스의 오른쪽 아래 꼭짓점 좌표 (픽셀 단위)

### 현재 상태 (임시 구현)
실제 AutoML 연동은 아직 구현되지 않았고, 테스트를 위해 **임의의 고정 좌표** `[50, 50, 400, 400]`을 반환하고 있습니다. 실제 서비스 시에는 `self.automl_endpoint.predict()`를 호출하여 실시간으로 좌표를 받아와야 합니다.

In [ ]:
    def _get_bounding_box(self, pet_img_bytes):
        """AutoML을 호출하여 강아지/고양이의 바운딩 박스를 가져옵니다."""
        # 실제 연동 시 self.automl_endpoint.predict()를 사용합니다.
        # 지금은 SAM 테스트를 위한 임의의 좌표(x_min, y_min, x_max, y_max)를 반환합니다.
        return np.array([50, 50, 400, 400])

## 4단계: `generate_fitting` — 핵심! 가상 피팅 실행 함수

이 메서드가 **실제로 반려동물에게 옷을 입히는 AI 작업을 수행하는 핵심 함수**입니다.
`main.py`에서 `pipeline.generate_fitting(pet_bytes, cloth_bytes)`로 호출됩니다.

### 매개변수
- `pet_img_bytes`: 반려동물 사진의 바이트 데이터 (URL에서 다운로드한 원본)
- `cloth_img_bytes`: 옷 사진의 바이트 데이터 (URL에서 다운로드한 원본)

---

### STEP 1: 바이트 → 이미지 변환 및 AutoML 호출

바이트 데이터를 AI 모델이 이해할 수 있는 **numpy 배열(숫자 행렬)**로 변환합니다.

- `cv2.imdecode()`: 바이트 데이터 → OpenCV 이미지(numpy 배열, BGR 색상)
- `cv2.cvtColor(..., cv2.COLOR_BGR2RGB)`: OpenCV는 색상을 BGR 순서로 다루지만, SAM과 PIL은 RGB 순서를 사용하므로 변환합니다.
- `_get_bounding_box()`: AutoML을 호출하여 반려동물의 위치(네모 박스 좌표)를 받아옵니다.

### STEP 2: SAM으로 마스크(실루엣) 추출

SAM에게 "이 네모 박스 안에 있는 물체의 정확한 윤곽선을 따줘!"라고 요청합니다.

- `set_image()`: SAM에 원본 이미지를 세팅합니다. (내부적으로 이미지의 특징을 미리 계산해둡니다)
- `predict(box=bbox)`: 바운딩 박스 좌표를 주면, 그 안의 물체(반려동물)의 **픽셀 단위 마스크**를 반환합니다.
- `multimask_output=False`: 마스크를 1개만 반환합니다. (True로 하면 3개의 후보 마스크를 반환)

> **마스크(Mask)란?** 원본 이미지와 같은 크기의 흑백 이미지입니다. 반려동물이 있는 픽셀은 **흰색(255)**, 배경 픽셀은 **검은색(0)**으로 표시됩니다. 이 마스크를 Stable Diffusion에 넘기면, "흰색 영역만 지우고 새로 그려라"는 의미가 됩니다.

### STEP 3: Stable Diffusion + IP-Adapter로 가상 피팅 합성

이 단계가 **최종 합성 작업**입니다. Stable Diffusion Inpainting 모델에 4가지 입력을 동시에 넣습니다:

| 입력 | 설명 |
|---|---|
| `prompt` | 텍스트 지시 — "반려동물이 옷을 입고 있는 사진, 고화질, 사실적" |
| `image` (pet_pil) | 원본 반려동물 사진 (PIL 이미지) |
| `mask_image` (mask_pil) | SAM이 만든 마스크 — 흰색(반려동물 영역)만 지우고 새로 그립니다 |
| `ip_adapter_image` (cloth_pil) | **핵심!** 옷 사진 — IP-Adapter가 이 이미지의 색상/패턴/질감을 추출하여 합성에 반영합니다 |

- `num_inference_steps=20`: Diffusion 모델이 노이즈(잡음)에서 깨끗한 이미지를 만들기까지의 반복 횟수입니다. 높을수록 품질이 좋아지지만 시간이 오래 걸립니다. (보통 20~50 사이)

### 최종 변환: PIL → JPEG 바이트
합성된 결과 이미지(PIL 객체)를 **JPEG 형식의 바이트 데이터**로 변환하여 반환합니다. `main.py`는 이 바이트 데이터를 GCS에 업로드합니다.

In [ ]:
    def generate_fitting(self, pet_img_bytes, cloth_img_bytes):
        # 1. 바이트 이미지를 OpenCV / Numpy 배열로 변환
        pet_img_np = cv2.imdecode(np.frombuffer(pet_img_bytes, np.uint8), cv2.IMREAD_COLOR)
        pet_img_rgb = cv2.cvtColor(pet_img_np, cv2.COLOR_BGR2RGB)
        
        # [STEP 1] AutoML: 위치 감지
        bbox = self._get_bounding_box(pet_img_bytes)

        # [STEP 2] SAM: 마스크(실루엣) 추출
        self.sam_predictor.set_image(pet_img_rgb)
        masks, _, _ = self.sam_predictor.predict(
            box=bbox,
            multimask_output=False
        )
        mask_image = masks[0]
        
        # Diffusion에 넣기 위해 PIL 이미지로 변환
        mask_pil = Image.fromarray((mask_image * 255).astype(np.uint8)).convert("L")
        pet_pil = Image.fromarray(pet_img_rgb)

        # 바이트 형태의 옷 사진을 PIL 이미지로 변환
        cloth_img_np = cv2.imdecode(np.frombuffer(cloth_img_bytes, np.uint8), cv2.IMREAD_COLOR)
        cloth_pil = Image.fromarray(cv2.cvtColor(cloth_img_np, cv2.COLOR_BGR2RGB))

        # [STEP 3] Diffusion: 가상 피팅 (Inpainting + IP-Adapter)
        prompt = "A pet wearing the target clothing, highly detailed, photorealistic"
        
        result_image = self.diffusion_pipe(
            prompt=prompt,
            image=pet_pil,
            mask_image=mask_pil,
            ip_adapter_image=cloth_pil,
            num_inference_steps=20
        ).images[0]

        # 최종 합성 이미지를 바이트로 변환하여 반환
        img_byte_arr = io.BytesIO()
        result_image.save(img_byte_arr, format='JPEG')
        return img_byte_arr.getvalue()

## 5단계: 파이프라인 인스턴스 생성 (싱글톤 패턴)

이 줄은 **파일이 import될 때 딱 한 번만 실행**됩니다.

`main.py`에서 `from vton_pipeline import pipeline`으로 이 파일을 불러올 때, 아래 코드가 자동으로 실행되어 `PetVTONPipeline()` 클래스의 인스턴스(객체)가 생성됩니다.

이때 `__init__` 함수가 호출되면서 3개의 AI 모델이 모두 메모리에 로드됩니다.

이후 `main.py`에서 `pipeline.generate_fitting()`을 호출하면, 이미 메모리에 올라가 있는 모델들이 즉시 사용됩니다.

> **싱글톤(Singleton) 패턴이란?** 프로그램 전체에서 특정 객체를 **단 하나만** 생성하여 공유하는 설계 패턴입니다. 여기서는 무거운 AI 모델을 중복 로딩하지 않기 위해 사용합니다.

In [ ]:
# 서버가 켜질 때 딱 한 번만 파이프라인 인스턴스 생성
pipeline = PetVTONPipeline()